# 03: Typology Profiling and Coverage Matrix
Profile each laundering typology by transaction code, channel, and risk attributes, then map typologies to scenarios to assess monitoring coverage and identify gaps.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)

DB_PATH = Path("../data/processed/aml.duckdb")
con = duckdb.connect(str(DB_PATH))

con.sql("SHOW TABLES").df()

,name
0,transactions
1,transactions_enriched
2,transactions_raw
3,txn_code_map


In [2]:
con.sql("""
    SELECT
        laundering_type,
        COUNT(*)                                                    AS n_txns,
        COUNT(DISTINCT sender_account)                              AS n_senders,
        ROUND(AVG(amount), 0)                                       AS avg_amount,
        ROUND(MEDIAN(amount), 0)                                    AS median_amount,
        ROUND(AVG(CASE WHEN is_cash             THEN 1 ELSE 0 END) * 100, 1) AS pct_cash,
        ROUND(AVG(CASE WHEN is_cross_border     THEN 1 ELSE 0 END) * 100, 1) AS pct_cross_border,
        ROUND(AVG(CASE WHEN is_foreign_currency THEN 1 ELSE 0 END) * 100, 1) AS pct_fx
    FROM transactions_enriched
    WHERE is_laundering = 1
    GROUP BY laundering_type
    ORDER BY n_txns DESC
""").df()

,laundering_type,n_txns,n_senders,avg_amount,median_amount,pct_cash,pct_cross_border,pct_fx
0,Structuring,1870,1870,5246.0,4800.0,0.0,38.7,50.4
1,Cash_Withdrawal,1334,88,154.0,139.0,100.0,30.9,30.9
2,Deposit-Send,945,563,9181.0,6332.0,50.1,35.8,36.9
3,Smurfing,932,63,2803.0,2629.0,100.0,0.5,0.5
4,Layered_Fan_In,656,491,17687.0,10726.0,0.0,19.7,28.5
5,Layered_Fan_Out,529,215,8629.0,6730.0,0.0,25.3,32.3
6,Stacked Bipartite,506,187,9727.0,7883.0,0.0,15.2,30.6
7,Behavioural_Change_1,394,53,4879.0,3118.0,0.0,65.5,66.8
8,Bipartite,383,74,11791.0,8901.0,0.0,24.0,41.0
9,Cycle,382,382,28026.0,21495.0,0.0,41.6,40.6


In [3]:
con.sql("""
    PIVOT (
        SELECT laundering_type, txn_code
        FROM transactions_enriched
        WHERE is_laundering = 1
    )
    ON txn_code
    USING COUNT(*)
    ORDER BY laundering_type
""").df()

,laundering_type,ACH,CC-POS,CHQ,CSH-DEP,CSH-WDL,DC-POS,WIR-INT
0,Behavioural_Change_1,42,29,42,0,0,23,258
1,Behavioural_Change_2,0,0,0,0,0,0,345
2,Bipartite,76,72,71,0,0,72,92
3,Cash_Withdrawal,0,0,0,0,1334,0,0
4,Cycle,63,62,51,0,0,47,159
5,Deposit-Send,25,35,36,473,0,38,338
6,Fan_In,79,85,72,0,0,82,46
7,Fan_Out,38,37,40,0,0,33,89
8,Gather-Scatter,76,73,62,0,0,93,50
9,Layered_Fan_In,146,125,125,0,0,131,129


In [4]:
con.sql("""
    SELECT
        COUNT(*)                            AS n_txns,
        COUNT(DISTINCT sender_account)      AS n_senders,
        COUNT(DISTINCT receiver_account)    AS n_receivers,
        MIN(amount)                         AS min_amount,
        QUANTILE_CONT(amount, 0.10)         AS p10,
        QUANTILE_CONT(amount, 0.50)         AS p50,
        QUANTILE_CONT(amount, 0.90)         AS p90,
        MAX(amount)                         AS max_amount
    FROM transactions_enriched
    WHERE laundering_type = 'Structuring'
""").df()

,n_txns,n_senders,n_receivers,min_amount,p10,p50,p90,max_amount
0,1870,1870,75,2078.6,2911.78,4800.35,8088.82,12454.4


In [6]:
con.sql("""
    SELECT
        CASE
            WHEN amount <  1000  THEN '1: < 1k'
            WHEN amount <  3000  THEN '2: 1k-3k'
            WHEN amount <  5000  THEN '3: 3k-5k'
            WHEN amount <  7500  THEN '4: 5k-7.5k'
            WHEN amount < 10000  THEN '5: 7.5k-10k'
            ELSE                      '6: 10k+'
        END AS amount_band,
        ROUND(SUM(CASE WHEN laundering_type = 'Structuring' THEN 1 ELSE 0 END) * 100.0
              / (SELECT COUNT(*) FROM transactions_enriched WHERE laundering_type = 'Structuring'), 1) AS pct_of_structuring,
        ROUND(SUM(CASE WHEN is_laundering = 0 THEN 1 ELSE 0 END) * 100.0
              / (SELECT COUNT(*) FROM transactions_enriched WHERE is_laundering = 0), 1)               AS pct_of_normal
    FROM transactions_enriched
    GROUP BY amount_band
    ORDER BY amount_band
""").df()

,amount_band,pct_of_structuring,pct_of_normal
0,1: < 1k,0.0,15.7
1,2: 1k-3k,11.8,14.0
2,3: 3k-5k,41.7,11.0
3,4: 5k-7.5k,30.4,19.0
4,5: 7.5k-10k,14.1,13.3
5,6: 10k+,2.0,26.9


In [7]:
con.sql("""
    WITH r AS (
        SELECT
            receiver_account,
            COUNT(*)                        AS n_struct_txns,
            COUNT(DISTINCT sender_account)  AS n_struct_senders,
            DATE_DIFF('day', MIN(txn_ts), MAX(txn_ts)) AS span_days
        FROM transactions_enriched
        WHERE laundering_type = 'Structuring'
        GROUP BY receiver_account
    )
    SELECT
        COUNT(*)                     AS n_receivers,
        MIN(n_struct_senders)        AS min_senders,
        MEDIAN(n_struct_senders)     AS median_senders,
        MAX(n_struct_senders)        AS max_senders,
        MEDIAN(span_days)            AS median_span_days,
        MAX(span_days)               AS max_span_days
    FROM r
""").df()

,n_receivers,min_senders,median_senders,max_senders,median_span_days,max_span_days
0,75,7,24.0,47,191.0,318


### Structuring in SAML-D
Structuring in this dataset does not follow the textbook cash-deposit pattern. It is entirely non-cash, spread across ACH, card, cheque, and wire payments, and each structuring sender appears exactly once. The pattern is many-to-few: 1,870 senders pay only 75 receiving accounts, roughly 25 senders per receiver. Amounts concentrate just below £5,000, with 41.7% of structuring falling between £3,000 and £5,000 compared with 11.0% of normal activity, and only 2% exceeding £10,000. A sender-level aggregation rule would therefore detect nothing. The scenario must instead aggregate at the receiver level, counting distinct senders paying sub-threshold amounts into the same account within a lookback window.

**Lookback window.** A typical structuring receiver collects payments from 24 distinct senders spread across roughly 191 days, or about four new senders per month, with the longest pattern spanning 318 days. This is a slow, low-and-slow pattern. A 30-day window would capture only a handful of senders per receiver, so 30, 60, and 90-day windows will be tested during threshold tuning, with the hypothesis that a longer window separates structuring from normal activity more effectively.

In [8]:
con.execute("""
    CREATE OR REPLACE TABLE coverage_matrix AS
    SELECT * FROM (VALUES
        ('Structuring',          'S1', 'Primary', 'Many senders pay one receiver, sub-5k amounts'),
        ('Smurfing',             'S2', 'Primary', '100% cash deposits, repeated by few accounts'),
        ('Cash_Withdrawal',      'S5', 'Primary', '100% cash withdrawals, frequent small amounts'),
        ('Deposit-Send',         'S3', 'Primary', 'Cash deposited then sent out, pass-through'),
        ('Gather-Scatter',       'S3', 'Primary', 'Funds gathered then dispersed, pass-through'),
        ('Behavioural_Change_2', 'S4', 'Primary', '100% cross-border and foreign currency'),
        ('Behavioural_Change_1', 'S4', 'Partial', '65% cross-border, rest domestic'),
        ('Fan_In',               'S1', 'Partial', 'Many-to-one, but not concentrated below 5k'),
        ('Layered_Fan_In',       'S1', 'Partial', 'Many-to-one through intermediary layers'),
        ('Fan_Out',              'S3', 'Partial', 'One-to-many, detectable only if funds pass through'),
        ('Layered_Fan_Out',      'S3', 'Partial', 'One-to-many through intermediary layers'),
        ('Scatter-Gather',       'S3', 'Partial', 'Dispersed then regathered across accounts'),
        ('Cycle',                'S3', 'Partial', 'Funds return to origin, needs network view'),
        ('Bipartite',            'S3', 'Partial', 'Two groups transacting, needs network view'),
        ('Stacked Bipartite',    'S3', 'Partial', 'Layered bipartite, needs network view'),
        ('Single_large',         NULL, 'Gap',     'One-off large payments overlap with normal activity'),
        ('Over-Invoicing',       NULL, 'Gap',     'Trade-based, requires invoice or trade data')
    ) AS t(laundering_type, primary_scenario, coverage_level, rationale)
""")

con.sql("SELECT * FROM coverage_matrix").df()

,laundering_type,primary_scenario,coverage_level,rationale
0,Structuring,S1,Primary,"Many senders pay one receiver, sub-5k amounts"
1,Smurfing,S2,Primary,"100% cash deposits, repeated by few accounts"
2,Cash_Withdrawal,S5,Primary,"100% cash withdrawals, frequent small amounts"
3,Deposit-Send,S3,Primary,"Cash deposited then sent out, pass-through"
4,Gather-Scatter,S3,Primary,"Funds gathered then dispersed, pass-through"
5,Behavioural_Change_2,S4,Primary,100% cross-border and foreign currency
6,Behavioural_Change_1,S4,Partial,"65% cross-border, rest domestic"
7,Fan_In,S1,Partial,"Many-to-one, but not concentrated below 5k"
8,Layered_Fan_In,S1,Partial,Many-to-one through intermediary layers
9,Fan_Out,S3,Partial,"One-to-many, detectable only if funds pass thr..."


In [9]:
con.sql("""
    WITH typology_counts AS (
        SELECT laundering_type, COUNT(*) AS n_laundering
        FROM transactions_enriched
        WHERE is_laundering = 1
        GROUP BY laundering_type
    )
    SELECT
        COALESCE(c.coverage_level, 'UNMAPPED')                     AS coverage_level,
        COUNT(*)                                                   AS n_typologies,
        SUM(t.n_laundering)                                        AS n_laundering_txns,
        ROUND(SUM(t.n_laundering) * 100.0
              / SUM(SUM(t.n_laundering)) OVER (), 1)               AS pct_of_laundering
    FROM typology_counts AS t
    LEFT JOIN coverage_matrix AS c
        ON t.laundering_type = c.laundering_type
    GROUP BY COALESCE(c.coverage_level, 'UNMAPPED')
    ORDER BY n_laundering_txns DESC
""").df()

,coverage_level,n_typologies,n_laundering_txns,pct_of_laundering
0,Primary,6,5780.0,58.5
1,Partial,9,3789.0,38.4
2,Gap,2,304.0,3.1


In [10]:
con.execute("""
    COPY coverage_matrix
    TO '../docs/typology_coverage_matrix.csv'
    (HEADER, DELIMITER ',')
""")
print("Saved to docs/typology_coverage_matrix.csv")

Saved to docs/typology_coverage_matrix.csv


### Coverage Assessment
All 17 laundering typologies map to the coverage matrix with no unmapped typologies. Weighted by laundering volume, 58.5% of laundering transactions fall under typologies with a dedicated primary scenario, 38.4% have partial coverage through a related scenario, and 3.1% represent documented gaps. The two gaps are Single_large, whose one-off high-value payments overlap heavily with normal activity, and Over-Invoicing, which is trade-based and cannot be monitored effectively without invoice or trade data. These figures reflect designed coverage, meaning what the scenarios are intended to detect. Effective coverage, meaning what they actually detect, will be measured once the scenarios are implemented and tested.

In [11]:
con.close()
print("Connection closed.")

Connection closed.
